# Segment an Object out of a Splat Scene — Standalone

A distilled, self-contained version of visergui's Segment tab
(`segmenter.py` + `multiview_mask.py` + `splat_trainer.select_splats_by_masks`).
Starts from a **click** — here a `(frame, u, v)` pixel instead of a live-viewer
pointer event — and ends with the scene split into `object.ply` + `background.ply`.

**Pipeline**

```
click (frame, u, v)
  └─> sample rendered splat depth -> backproject to world point P
        └─> seed frame = closest training camera that sees P
              └─> SAM 2 image predictor: point prompt -> seed mask (preview)
                    └─> SAM 2 video predictor: propagate the mask to ALL frames
                          └─> multi-view front-surface vote -> selected splats
                                └─> object.ply + background.ply + renders
```

Inputs come from `splat_init_and_train.ipynb`'s `_work/`: a splat PLY plus
`cameras.npz` (frames, K, poses, depth) — no DA3 needed here. Requires the
`splat` conda env, `sam2`, and the SAM 2.1 hiera-large checkpoint.


In [ ]:
from pathlib import Path

# ---- scene (artifacts from splat_init_and_train.ipynb) ----
SPLAT_PLY   = "_work/splats_trained.ply"   # scene to segment (or splats_voxel.ply)
CAMERAS_NPZ = "_work/cameras.npz"          # frames + cameras saved by notebook 1

# ---- the "click": a pixel in a chosen frame ----
# In the GUI this is a viser pointer event on the live view (segmenter.py:228).
# Coordinates are in the DA3 processed-image pixel space of cameras.npz frames.
CLICK_FRAME = 0
CLICK_U     = 252
CLICK_V     = 200

# ---- SAM 2 ----
SAM2_CKPT = "../../vendor/InstaInpaint/checkpoints/sam2.1_hiera_large.pt"
SAM2_CFG  = "configs/sam2.1/sam2.1_hiera_l.yaml"  # hydra name inside the sam2 package

# ---- splat-selection vote (splat_trainer.py:1713 defaults) ----
FRAC      = 0.5    # selected if in-mask in >= this fraction of the frames that see it
MIN_SEEN  = 1      # must be seen by at least this many frames
DEPTH_TOL = 0.06   # relative tolerance for the front-surface depth gate

DEVICE  = "cuda"
OUT_DIR = "_work/segment"


In [ ]:
import shutil
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from plyfile import PlyData, PlyElement

OUT_DIR = Path(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device(DEVICE if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

# scene frames + cameras (saved by notebook 1 — no DA3 run needed)
npz = np.load(CAMERAS_NPZ)
rgb_u8 = npz["rgb"]                                   # (N,H,W,3) uint8
K_np, c2w_np = npz["K"], npz["c2w"]                   # (N,3,3), (N,4,4) OpenCV
w2c = torch.from_numpy(npz["w2c"]).to(device)
K_t = torch.from_numpy(K_np).to(device)
N, H, W = rgb_u8.shape[:3]
print(f"{N} frames @ {W}x{H}")


def load_inria_ply(path, device):
    """Inverse of notebook 1's save_inria_ply (splat_trainer.py:950): returns the
    RAW parameters (log-scales, logit-opacities, unnormalized quats); the SH
    degree is inferred from the number of f_rest_* fields."""
    v = PlyData.read(str(path))["vertex"].data
    M = len(v)

    def cols(*names):
        return torch.from_numpy(
            np.stack([np.asarray(v[n], np.float32) for n in names], 1)).to(device)

    n_rest = len([n for n in v.dtype.names if n.startswith("f_rest_")])
    K_rest = n_rest // 3
    if K_rest:
        rest = np.stack([np.asarray(v[f"f_rest_{i}"], np.float32)
                         for i in range(n_rest)], 1)
        shN = torch.from_numpy(
            rest.reshape(M, 3, K_rest).transpose(0, 2, 1).copy()).to(device)
    else:
        shN = torch.zeros((M, 0, 3), device=device)
    return {
        "means":     cols("x", "y", "z"),
        "scales":    cols("scale_0", "scale_1", "scale_2"),
        "quats":     cols("rot_0", "rot_1", "rot_2", "rot_3"),
        "opacities": torch.from_numpy(np.asarray(v["opacity"], np.float32)).to(device),
        "sh0":       cols("f_dc_0", "f_dc_1", "f_dc_2")[:, None, :],
        "shN":       shN,
    }


params = load_inria_ply(SPLAT_PLY, device)
M = params["means"].shape[0]
print(f"{M} gaussians from {SPLAT_PLY} ({params['shN'].shape[1]} SH rest coeffs)")


## Rendering and camera helpers

`render_rgbd` renders the splats at any OpenCV camera and recovers metric depth
from gsplat's `RGB+ED` output (`splat_trainer._render_splat_rgbd`, :410-428).
The two projection helpers are `multiview_mask.py:97-125` verbatim.


In [ ]:
def render_rgbd(params, c2w_i, K_i, H, W):
    from gsplat import rasterization
    dev = params["means"].device
    c2w_t = torch.as_tensor(c2w_i, dtype=torch.float32, device=dev).reshape(4, 4)
    K_cam = torch.as_tensor(K_i, dtype=torch.float32, device=dev).reshape(3, 3)
    with torch.no_grad():
        sh = torch.cat([params["sh0"], params["shN"]], dim=1)
        sh_deg = int(round(sh.shape[1] ** 0.5)) - 1
        out, alpha, _ = rasterization(
            means=params["means"], quats=F.normalize(params["quats"], dim=-1),
            scales=torch.exp(params["scales"]),
            opacities=torch.sigmoid(params["opacities"]),
            colors=sh, viewmats=torch.linalg.inv(c2w_t)[None], Ks=K_cam[None],
            width=int(W), height=int(H), sh_degree=sh_deg, packed=False,
            render_mode="RGB+ED")
        a = alpha[0, :, :, 0].clamp(0, 1)
        rgb = (out[0, :, :, :3].clamp(0, 1).cpu().numpy() * 255).astype(np.uint8)
        depth = (out[0, :, :, 3] / a.clamp_min(1e-6)).cpu().numpy()
    return rgb, a.cpu().numpy(), depth


def backproject_to_world(u, v, depth, K_i, c2w_i):
    """Pixel (u,v) + depth -> world point, OpenCV convention."""
    fx, fy = float(K_i[0, 0]), float(K_i[1, 1])
    cx, cy = float(K_i[0, 2]), float(K_i[1, 2])
    p_cam = np.array([(u - cx) / fx * depth, (v - cy) / fy * depth, depth, 1.0],
                     dtype=np.float32)
    return (c2w_i @ p_cam)[:3]


def project_world_to_pixel(p_world, K_i, c2w_i):
    """World point -> (u, v, z_camera); z > 0 means in front of the camera."""
    w2c_i = np.linalg.inv(c2w_i)
    p_cam = (w2c_i @ np.append(p_world, 1.0).astype(np.float32))[:3]
    z = float(p_cam[2])
    if z <= 1e-6:
        return float("nan"), float("nan"), z
    u = float(K_i[0, 0] * p_cam[0] / z + K_i[0, 2])
    v = float(K_i[1, 1] * p_cam[1] / z + K_i[1, 2])
    return u, v, z


def overlay_mask(rgb, mask, color=(255, 0, 0), alpha=0.5):
    out = rgb.astype(np.float32).copy()
    out[mask] = (1 - alpha) * out[mask] + alpha * np.array(color, np.float32)
    return out.astype(np.uint8)


## 1 — The click: pixel → world point → seed frame

Exactly what the GUI does with a pointer event (`segmenter.py:228-275`): sample
the **rendered splat depth** at the clicked pixel (rejecting empty space via the
alpha channel), backproject to a world point `P`, then pick the **seed frame** as
the training camera closest to `P` among those that see it (`:161-182` — closest
camera ≈ largest apparent size, most likely unoccluded).


In [ ]:
_, alpha_c, depth_c = render_rgbd(params, c2w_np[CLICK_FRAME], K_np[CLICK_FRAME], H, W)
z_click = float(depth_c[CLICK_V, CLICK_U])
assert z_click > 0 and float(alpha_c[CLICK_V, CLICK_U]) >= 0.5, (
    f"clicked empty space at frame {CLICK_FRAME} ({CLICK_U},{CLICK_V}) — "
    f"alpha={float(alpha_c[CLICK_V, CLICK_U]):.2f}; pick a pixel on the object")
P = backproject_to_world(CLICK_U, CLICK_V, z_click, K_np[CLICK_FRAME], c2w_np[CLICK_FRAME])

best, best_dist = None, float("inf")
for i in range(N):
    u, v, z = project_world_to_pixel(P, K_np[i], c2w_np[i])
    if not (z > 0) or np.isnan(u):
        continue
    if not (0 <= u < W and 0 <= v < H):
        continue
    dist = float(np.linalg.norm(c2w_np[i][:3, 3] - P))
    if dist < best_dist:
        best_dist, best = dist, (i, float(u), float(v))
assert best is not None, "no training frame sees the clicked point"
seed_i, su, sv = best
print(f"P = {np.round(P, 3).tolist()}  ->  seed frame {seed_i}, pixel ({su:.0f}, {sv:.0f})")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(rgb_u8[CLICK_FRAME])
axes[0].scatter([CLICK_U], [CLICK_V], c="r", marker="x", s=100)
axes[0].set_title(f"click: frame {CLICK_FRAME} ({CLICK_U},{CLICK_V})")
axes[1].imshow(rgb_u8[seed_i])
axes[1].scatter([su], [sv], c="r", marker="x", s=100)
axes[1].set_title(f"seed frame {seed_i} ({su:.0f},{sv:.0f})")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
fig.savefig(OUT_DIR / "click_seed.png", dpi=110)
plt.show()


## 2 — SAM 2 seed mask (single frame)

One positive point prompt on the seed frame, best of the three multimask outputs
(`segmenter.py:184-195`). This is the preview the GUI shows before propagating.


In [ ]:
def build_sam2_predictor(ckpt, cfg, device, video=False):
    """multiview_mask.py:40-82. cfg is a hydra config name resolved inside the
    sam2 package. sam2/__init__.py initializes hydra's global state only on
    first import, so clear + re-initialize here to make every build (and cell
    re-run) start from the same clean state."""
    from hydra import initialize_config_module
    from hydra.core.global_hydra import GlobalHydra
    GlobalHydra.instance().clear()
    initialize_config_module("sam2", version_base="1.2")
    assert Path(ckpt).exists(), (
        f"SAM2 checkpoint missing: {ckpt}\ndownload: "
        "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt")
    if video:
        from sam2.build_sam import build_sam2_video_predictor
        return build_sam2_video_predictor(cfg, str(ckpt), device=str(device))
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    return SAM2ImagePredictor(build_sam2(cfg, str(ckpt), device=str(device)))


img_predictor = build_sam2_predictor(SAM2_CKPT, SAM2_CFG, device)
with torch.inference_mode():
    img_predictor.set_image(rgb_u8[seed_i])
    m_pred, scores, _ = img_predictor.predict(
        point_coords=np.array([[su, sv]], dtype=np.float32),
        point_labels=np.array([1], dtype=np.int32),
        multimask_output=True)
seed_mask = m_pred[int(np.argmax(scores))].astype(bool)
del img_predictor
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"seed mask covers {100 * seed_mask.mean():.1f}% of frame {seed_i}")
plt.figure(figsize=(7, 4.5))
plt.imshow(overlay_mask(rgb_u8[seed_i], seed_mask))
plt.title(f"SAM 2 seed mask (frame {seed_i})")
plt.axis("off")
plt.savefig(OUT_DIR / "seed_mask.png", dpi=110, bbox_inches="tight")
plt.show()


## 3 — Propagate to every frame (SAM 2 video predictor)

The training frames are treated as a video: dump them as numbered JPEGs, seed the
video predictor with the same point at the seed frame, and propagate **both
directions** (`segmenter.py:300-337`). Purely temporal — no 3D involved.


In [ ]:
tmpdir = Path(tempfile.mkdtemp(prefix="sam2_frames_"))
masks = np.zeros((N, H, W), dtype=bool)
try:
    for i in range(N):
        Image.fromarray(rgb_u8[i]).save(str(tmpdir / f"{i:05d}.jpg"), quality=95)

    video_predictor = build_sam2_predictor(SAM2_CKPT, SAM2_CFG, device, video=True)
    autocast = torch.autocast(
        device_type="cuda" if device.type == "cuda" else "cpu", dtype=torch.bfloat16)
    with torch.inference_mode(), autocast:
        state = video_predictor.init_state(video_path=str(tmpdir),
                                           offload_video_to_cpu=True)
        video_predictor.add_new_points_or_box(
            inference_state=state, frame_idx=seed_i, obj_id=1,
            points=np.array([[su, sv]], dtype=np.float32),
            labels=np.array([1], dtype=np.int32))
        for reverse in (False, True):
            for fidx, _obj_ids, logits in video_predictor.propagate_in_video(
                    state, reverse=reverse):
                m = (logits[0, 0] > 0.0).cpu().numpy()
                if m.shape != (H, W):   # SAM2 may work at its own internal res
                    m = np.asarray(Image.fromarray(m.astype(np.uint8) * 255)
                                   .resize((W, H), Image.NEAREST)) > 127
                masks[fidx] = m
    del video_predictor, state
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
finally:
    shutil.rmtree(tmpdir, ignore_errors=True)

for i in range(N):
    Image.fromarray(masks[i].astype(np.uint8) * 255).save(
        str(OUT_DIR / f"objmask_{i:04d}.png"))
n_nonempty = int(masks.reshape(N, -1).any(axis=1).sum())
print(f"object mask present in {n_nonempty}/{N} frames "
      f"({100 * masks.mean():.1f}% of all pixels)")

cols_n = min(4, N)
rows_n = (N + cols_n - 1) // cols_n
fig, axes = plt.subplots(rows_n, cols_n, figsize=(4 * cols_n, 2.6 * rows_n),
                         squeeze=False)
for i in range(rows_n * cols_n):
    ax = axes[i // cols_n, i % cols_n]
    if i < N:
        ax.imshow(overlay_mask(rgb_u8[i], masks[i]))
        ax.set_title(f"frame {i}", fontsize=9)
    ax.axis("off")
fig.tight_layout()
fig.savefig(OUT_DIR / "masks_montage.png", dpi=110)
plt.show()


## 4 — Lift masks to 3D: multi-view front-surface vote

Distills `SplatTrainer.select_splats_by_masks` (splat_trainer.py:1713-1798).
For each frame: project every gaussian **center**; a splat is *seen* if it's in
front of the camera, in bounds, and its camera-space depth matches the rendered
splat depth at that pixel within `DEPTH_TOL` (relative) — i.e. it belongs to the
visible front surface, so hidden splats behind the object can't vote. A splat is
selected if it lands inside the mask in ≥ `FRAC` of the frames that saw it.


In [ ]:
means = params["means"].detach()
means_h = torch.cat([means, torch.ones((M, 1), device=device)], dim=1)  # (M,4)
votes_seen = torch.zeros(M, dtype=torch.int32, device=device)
votes_in = torch.zeros(M, dtype=torch.int32, device=device)
masks_t = torch.from_numpy(masks).to(device)

with torch.no_grad():
    for i in range(N):
        p_cam = (w2c[i] @ means_h.T).T[:, :3]
        z = p_cam[:, 2]
        z_safe = z.clamp_min(1e-6)
        fx, fy = K_t[i, 0, 0], K_t[i, 1, 1]
        cx, cy = K_t[i, 0, 2], K_t[i, 1, 2]
        pix_x = fx * (p_cam[:, 0] / z_safe) + cx
        pix_y = fy * (p_cam[:, 1] / z_safe) + cy
        in_front = z > 0
        in_bounds = (pix_x >= 0) & (pix_x < W) & (pix_y >= 0) & (pix_y < H)

        _, _, depth_np = render_rgbd(params, c2w_np[i], K_np[i], H, W)
        depth_r = torch.as_tensor(depth_np, device=device)
        px = pix_x.clamp(0, W - 1).long()
        py = pix_y.clamp(0, H - 1).long()
        rendered = depth_r[py, px]                       # (M,)
        in_mask_px = masks_t[i][py, px]                  # (M,) bool
        front = (z - rendered).abs() <= DEPTH_TOL * z_safe
        seen = in_front & in_bounds & (rendered > 0) & front
        votes_seen += seen.to(torch.int32)
        votes_in += (seen & in_mask_px).to(torch.int32)

selected = (votes_seen >= MIN_SEEN) & (votes_in.float() >= FRAC * votes_seen.float())
n_sel = int(selected.sum())
print(f"selected {n_sel} / {M} splats ({100 * n_sel / M:.1f}%)")
assert 0 < n_sel < M, "selection degenerate — tune the click point or FRAC"


## 5 — Split the scene: object + background PLYs, and proof renders

The GUI stops at highlighting; here we finish the job — the selection splits the
parameter set row-wise into two standard Inria PLYs, then we re-render the seed
view three ways to show the cut is clean.


In [ ]:
def save_inria_ply(path, means, sh_all, log_s, logit_o, quats):
    """Inria 3DGS PLY writer (splat_trainer.py:920), same as notebook 1."""
    n = means.shape[0]
    K_rest = sh_all.shape[1] - 1
    fields = ([("x", "f4"), ("y", "f4"), ("z", "f4"),
               ("nx", "f4"), ("ny", "f4"), ("nz", "f4"),
               ("f_dc_0", "f4"), ("f_dc_1", "f4"), ("f_dc_2", "f4")]
              + [(f"f_rest_{i}", "f4") for i in range(3 * K_rest)]
              + [("opacity", "f4"),
                 ("scale_0", "f4"), ("scale_1", "f4"), ("scale_2", "f4"),
                 ("rot_0", "f4"), ("rot_1", "f4"), ("rot_2", "f4"), ("rot_3", "f4")])
    arr = np.zeros(n, dtype=fields)
    m = means.detach().cpu().numpy()
    sh_np = sh_all.detach().cpu().numpy()
    s = log_s.detach().cpu().numpy()
    q = quats.detach().cpu().numpy()
    arr["x"], arr["y"], arr["z"] = m[:, 0], m[:, 1], m[:, 2]
    arr["f_dc_0"], arr["f_dc_1"], arr["f_dc_2"] = sh_np[:, 0, 0], sh_np[:, 0, 1], sh_np[:, 0, 2]
    if K_rest > 0:
        rest = sh_np[:, 1:, :].transpose(0, 2, 1).reshape(n, 3 * K_rest)
        for i in range(3 * K_rest):
            arr[f"f_rest_{i}"] = rest[:, i]
    arr["opacity"] = logit_o.detach().cpu().numpy()
    arr["scale_0"], arr["scale_1"], arr["scale_2"] = s[:, 0], s[:, 1], s[:, 2]
    arr["rot_0"], arr["rot_1"], arr["rot_2"], arr["rot_3"] = q[:, 0], q[:, 1], q[:, 2], q[:, 3]
    PlyData([PlyElement.describe(arr, "vertex")], text=False).write(str(path))


def subset(p, keep):
    return {k: v[keep] for k, v in p.items()}


for name, keep in (("object", selected), ("background", ~selected)):
    part = subset(params, keep)
    save_inria_ply(OUT_DIR / f"{name}.ply", part["means"],
                   torch.cat([part["sh0"], part["shN"]], dim=1),
                   part["scales"], part["opacities"], part["quats"])
    print(f"saved {OUT_DIR / (name + '.ply')}  ({int(keep.sum())} gaussians)")

r_full, _, _ = render_rgbd(params, c2w_np[seed_i], K_np[seed_i], H, W)
r_obj, _, _ = render_rgbd(subset(params, selected), c2w_np[seed_i], K_np[seed_i], H, W)
r_bg, _, _ = render_rgbd(subset(params, ~selected), c2w_np[seed_i], K_np[seed_i], H, W)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, img, title in zip(axes, (r_full, r_obj, r_bg),
                          ("full scene", "object only", "background only")):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")
fig.tight_layout()
fig.savefig(OUT_DIR / "segment_result.png", dpi=110)
plt.show()


## Notes

- **The click** is the only thing that differs from the GUI: viser hands the demo a
  normalized viewport position on a live free camera (`segmenter.py:228-275`); here
  it's a pixel in a training frame, which follows the identical path afterward
  (depth sample → backproject → seed-frame pick).
- **Vote knobs**: raise `FRAC` (e.g. 0.7) for a stricter object cut; raise
  `MIN_SEEN` to drop splats seen in only one view; `DEPTH_TOL` is a *relative*
  front-surface gate — looser values let occluded splats vote.
- The vote tests gaussian **centers** only — big boundary splats straddling the
  silhouette can end up on either side. Train longer (smaller splats) for a
  crisper cut.
- To highlight instead of split, keep `selected` and color those splats in a viewer
  — that's all the GUI's yellow `selected_object` overlay does (`segmenter.py:417-448`).
